# Scheduling Recommendations
**MODEL-DRIVEN SCHEDULE OPTIMIZATION · RISK MATRIX · BUFFER RECOMMENDATIONS**

---


## Inhalt

- [The Full Circle](#the-full-circle)
- [Setup](#setup)
- [Risiko-Matrix: Stop × Linie × Stunde × Kontext](#risiko-matrix-stop-linie-stunde-kontext)
- [Top Recommendations — Priority List](#top-recommendations-priority-list)
  - [Konkretes Beispiel: Wie wird aus einem Modell-Output eine Empfehlung?](#konkretes-beispiel-wie-wird-aus-einem-modell-output-eine-empfehlung)
- [Kontext-Vergleich: Normal vs. Schnee vs. Event](#kontext-vergleich-normal-vs-schnee-vs-event)
- [Karte: Wo braucht es Puffer?](#karte-wo-braucht-es-puffer)
- [Scope: How Large Is the Need for Action?](#scope-how-large-is-the-need-for-action)
- [Key Findings](#key-findings)
- [Export](#export)


## The Full Circle

```
1. Analyse       → Delays identifizieren und messen
                   (66 strukturierte Findings, 6 Dimensionen)

2. Kernbefund    → Delays sind intrinsisch, nicht zufällig:
                   dwell_time = Feature #1 · Kaskadeneffekt r=0.85
                   71% aller Halte ohne Puffer → System kann Verspätung
                   nicht abbauen, nur weitergeben

3. Modell        → Vorhersage MAE 18.56s beweist Strukturalität:
                   Zufällige Delays sind nicht vorhersagbar.
                   Vorhersagbar = strukturell = steuerbar.

4. Empfehlung    → Modell-Outputs als Input für Fahrplandesign:
                   WO und WANN entstehen hohe Delays? → dort Puffer einplanen.
```

**Dieses Notebook:** Berechnet eine datengetriebene Empfehlungstabelle —
welche Haltestellen, auf welchen Linien, zu welchen Stunden und unter
welchen Bedingungen einen Fahrplan-Puffer brauchen.

**Wichtige Einschränkung (aus `06_prediction_6`):**
Das Modell sagt *wo und wann* das Risiko hoch ist — nicht *wie viel* Puffer optimal wäre.
Puffergrößen sind Startpunkte für operatives Testing, keine Modellausgaben.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import polars as pl
import lightgbm as lgb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import json
from pathlib import Path

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("06_prediction_7-recommendations")

MODELS_DIR = Path(TRAIN).parent.parent / "models"
%load_ext autoreload
%autoreload 2

In [ ]:
# lgbm_v1: schedule-based features only (no prev_trip_delay)
# → richtig für Fahrplanplanung: alle Features zum Planungszeitpunkt bekannt
# lgbm_v2 nutzt prev_trip_delay — das ist ein Echtzeitfeature, nicht planbar
model = lgb.Booster(model_file=str(MODELS_DIR / "lgbm_v1.txt"))

with open(MODELS_DIR / "lgbm_v1_meta.json") as f:
    meta = json.load(f)

FEATURE_COLS = meta["features"]
CAT_COLS = meta["cat_cols"]

# test_final.parquet hat alle Modell-Features (gtfs_year, dwell_time, n_lines_at_stop, etc.)
# test_features.parquet ist die Zwischen-Stufe vor dem Feature-Engineering → hier NICHT verwenden
TEST_FINAL = PATHS["processed"] / "test_final.parquet"

# Polars Lernmoment: lazy loading + cast in einer Pipeline, collect erst am Ende
df_test = (
    pl.scan_parquet(TEST_FINAL)
    .with_columns(pl.col("line_name").cast(pl.Utf8))
    .collect()
    .to_pandas()
)
for col in CAT_COLS:
    df_test[col] = df_test[col].astype("category")

df_test["pred"] = model.predict(df_test[FEATURE_COLS])

# Network reference values
NETWORK_MEAN = df_test["arrival_delay"].mean()
BUFFER_THRESHOLD = 60.0  # Stops with predicted delay above this get a buffer recommendation

print(f"Test rows: {len(df_test):,}")
print(f"Network Ø delay: {NETWORK_MEAN:.1f}s")
print(f"Buffer threshold: {BUFFER_THRESHOLD}s (above = buffer recommended)")

## Risiko-Matrix: Stop × Linie × Stunde × Kontext

Für jede Kombination aus Haltestelle, Linie, Tageszeit und Betriebsbedingung
berechnet das Modell einen mittleren Delay. Das ist die Grundlage der Empfehlung.

**Kontexte:**
- `Normal` — Werktag, kein Schnee, kein Event
- `Schnee` — has_snow = True
- `Event` — has_event = True
- `Rush` — Donnerstag/Freitag 17–19h (schlechtester Peak laut Temporal-Analyse)
- `Spätnacht` — 21h+ (Post-Event-Welle)

In [ ]:
def assign_context(df: pd.DataFrame) -> pd.DataFrame:
    """Assign operational context label to each row."""
    df = df.copy()
    conditions = [
        df["has_snow"].astype(bool),
        df["has_event"].astype(bool) & (df["hour"] >= 18) & (df["hour"] <= 22),
        (df["weekday"].isin([3, 4])) & (df["hour"] >= 17) & (df["hour"] <= 19),
        df["hour"] >= 21,
    ]
    choices = ["Schnee", "Event", "Rush", "Spätnacht"]
    df["context"] = np.select(conditions, choices, default="Normal")
    return df


df_test = assign_context(df_test)
print(df_test["context"].value_counts().to_string())

In [ ]:
# Aggregate: mean predicted delay per (stop, line, context)
# min_n=500 filtert statistisch instabile Gruppen
risk_matrix = (
    df_test
    .groupby(["stop_name", "line_name", "context"], observed=True)
    .agg(
        pred_delay=("pred", "mean"),
        actual_delay=("arrival_delay", "mean"),
        stop_lat=("stop_lat", "mean"),
        stop_lon=("stop_lon", "mean"),
        n=("pred", "count"),
    )
    .reset_index()
    .query("n >= 500")
)

# Buffer recommendation
risk_matrix["buffer_needed"] = risk_matrix["pred_delay"] > BUFFER_THRESHOLD
risk_matrix["excess_delay"] = (risk_matrix["pred_delay"] - BUFFER_THRESHOLD).clip(lower=0)

# Rough buffer heuristic: 1/3 of excess delay, rounded to 5s, capped at 60s
# Rationale: not all excess can be recovered at a single stop
risk_matrix["buffer_rec_s"] = (
    (risk_matrix["excess_delay"] / 3)
    .apply(lambda x: round(x / 5) * 5)
    .clip(upper=60)
    .where(risk_matrix["buffer_needed"], other=0)
    .astype(int)
)

n_flagged = risk_matrix["buffer_needed"].sum()
n_total = len(risk_matrix)
print(f"Stop-Linie-Kontext Kombinationen: {n_total:,}")
print(f"Davon mit Buffer-Empfehlung (pred > {BUFFER_THRESHOLD}s): {n_flagged:,} ({n_flagged/n_total:.1%})")

## Top Recommendations — Priority List


In [ ]:
top_recs = (
    risk_matrix[risk_matrix["buffer_needed"]]
    .sort_values("pred_delay", ascending=False)
    .head(20)
    .copy()
)

top_recs["Haltestelle"] = top_recs["stop_name"].str.replace("Zürich, ", "", regex=False)
top_recs["Linie"] = top_recs["line_name"].apply(lambda x: f"L{x}")
top_recs["Ø Pred. Delay (s)"] = top_recs["pred_delay"].round(1)
top_recs["Ø Ist Delay (s)"] = top_recs["actual_delay"].round(1)
top_recs["Kontext"] = top_recs["context"]
top_recs["Buffer-Empfehlung (s)"] = top_recs["buffer_rec_s"]
top_recs["N"] = top_recs["n"]

show_df(
    top_recs[["Haltestelle", "Linie", "Kontext", "Ø Pred. Delay (s)", "Ø Ist Delay (s)", "Buffer-Empfehlung (s)", "N"]]
    .reset_index(drop=True)
)

### Konkretes Beispiel: Wie wird aus einem Modell-Output eine Empfehlung?

Nehmen wir **Haltestelle Friedhof Enzenbühl · Linie 11 · Kontext: Normal**:

1. **Analyse** (F-SPAT-01): Friedhof Enzenbühl hat historisch 93.8s Ø Delay — höchster Wert im Netz.
2. **Modell** (lgbm_v1): Predicted Delay für diese Stop-Linie-Kontext-Kombination = ~85–95s
3. **Schwelle**: Buffer-Threshold = 60s → pred. Delay liegt weit darüber
4. **Heuristik**: Buffer = 1/3 × (pred. Delay − 60s), gerundet auf 5s, max. 60s
   - Beispiel: pred. = 90s → Überschuss = 30s → Buffer = 10s
5. **Empfehlung**: "Haltestelle Friedhof Enzenbühl auf Linie 11 benötigt im Normalbetrieb
   10s zusätzlichen Fahrplan-Puffer"

**Für Schnee-Kontext** ändert sich das Bild: Das Modell sagt für denselben Stop bei Schnee
deutlich höhere Delays voraus — die Empfehlung würde auf z.B. 30s steigen. Das ist der Punkt
von `F-REC-03`: ein einheitlicher Puffer für alle Kontexte greift zu kurz.

**Was das Modell nicht kann:** Den kausalen Effekt eines +10s Puffers messen.
Ob der Puffer tatsächlich 10s spart, muss durch ein operatives A/B-Experiment bestätigt werden.
Das Modell liefert die *Diagnose*, der Betrieb liefert die *Dosis*.

## Kontext-Vergleich: Normal vs. Schnee vs. Event

Wie verändert sich das Risikobild je nach Betriebsbedingung?
Zeigt welche Linien/Stops kontextspezifische Puffer brauchen.

In [ ]:
# Per-line, per-context: mean predicted delay
line_context = (
    risk_matrix
    .groupby(["line_name", "context"], observed=True)["pred_delay"]
    .mean()
    .reset_index()
)

contexts = ["Normal", "Rush", "Event", "Schnee", "Spätnacht"]
colors   = ["#2E86AB", "#ffa600", "#de425b", "#6a5acd", "#25ac82"]

fig = go.Figure()

for ctx, col in zip(contexts, colors):
    sub = line_context[line_context["context"] == ctx].sort_values("pred_delay", ascending=False)
    fig.add_trace(go.Bar(
        name=ctx,
        x=sub["line_name"].apply(lambda x: f"L{x}"),
        y=sub["pred_delay"].round(1),
        marker_color=col,
    ))

fig.add_hline(
    y=BUFFER_THRESHOLD,
    line_dash="dot",
    line_color="#888",
    annotation_text=f"Buffer-Schwelle {BUFFER_THRESHOLD}s",
    annotation_position="top right",
)

fig.update_layout(
    barmode="group",
    title=dict(
        text="Ø Pred. Delay nach Linie × Kontext — Wann welche Linie Puffer braucht",
        x=0, xanchor="left",
    ),
    xaxis=dict(title="Linie"),
    yaxis=dict(title="Ø Pred. Delay (s)"),
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=0, r=0, t=100, b=40),
    plot_bgcolor="white",
)
fig.show()

# Table: lines that need buffer in multiple contexts
buffer_by_line_ctx = (
    risk_matrix[risk_matrix["buffer_needed"]]
    .groupby(["line_name", "context"], observed=True)
    .agg(n_stops=("stop_name", "nunique"), mean_excess=("excess_delay", "mean"))
    .reset_index()
)
buffer_by_line_ctx["Linie"] = buffer_by_line_ctx["line_name"].apply(lambda x: f"L{x}")
buffer_by_line_ctx["Kontext"] = buffer_by_line_ctx["context"]
buffer_by_line_ctx["Stops mit Buffer-Bedarf"] = buffer_by_line_ctx["n_stops"]
buffer_by_line_ctx["Ø Überschuss (s)"] = buffer_by_line_ctx["mean_excess"].round(1)
show_df(
    buffer_by_line_ctx[["Linie", "Kontext", "Stops mit Buffer-Bedarf", "Ø Überschuss (s)"]]
    .sort_values(["Linie", "Kontext"])
    .reset_index(drop=True)
)

## Karte: Wo braucht es Puffer?

Alle Haltestellen mit Buffer-Empfehlung — Farbe nach Kontext, Größe nach empfohlenem Puffer.
Filter auf Normal-Kontext um den strukturellen Basisbedarf zu zeigen.

In [ ]:
ctx_colors = {
    "Normal":    "#2E86AB",
    "Rush":      "#ffa600",
    "Event":     "#de425b",
    "Schnee":    "#6a5acd",
    "Spätnacht": "#25ac82",
}

flagged = risk_matrix[risk_matrix["buffer_needed"]].copy()
flagged["stop_short"] = flagged["stop_name"].str.replace("Zürich, ", "", regex=False)
flagged["bubble"] = 8 + (flagged["buffer_rec_s"] / 60) * 20  # scale 8–28px

fig = go.Figure()

for ctx, col in ctx_colors.items():
    sub = flagged[flagged["context"] == ctx]
    if sub.empty:
        continue
    fig.add_trace(go.Scattermapbox(
        lat=sub["stop_lat"],
        lon=sub["stop_lon"],
        mode="markers",
        name=ctx,
        marker=dict(size=sub["bubble"], color=col, opacity=0.80),
        customdata=sub[["stop_short", "line_name", "pred_delay", "buffer_rec_s"]].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b> · L%{customdata[1]}<br>"
            "Pred. Delay: <b>%{customdata[2]:.0f}s</b><br>"
            "Buffer-Empfehlung: <b>+%{customdata[3]}s</b>"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    mapbox=dict(
        style="carto-positron",
        center=dict(lat=47.378, lon=8.540),
        zoom=11.5,
    ),
    title=dict(
        text="Fahrplan-Puffer-Empfehlungen nach Haltestelle und Kontext<br><sup>Größe = empfohlener Puffer · Farbe = Betriebskontext</sup>",
        x=0, xanchor="left",
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    height=600,
    margin=dict(l=0, r=0, t=80, b=0),
)
fig.show()

## Scope: How Large Is the Need for Action?


In [ ]:
# Summary: how many unique stops need a buffer, per context?
scope = (
    risk_matrix[risk_matrix["buffer_needed"]]
    .groupby("context", observed=True)
    .agg(
        unique_stops=("stop_name", "nunique"),
        unique_lines=("line_name", "nunique"),
        mean_buffer=("buffer_rec_s", "mean"),
        max_buffer=("buffer_rec_s", "max"),
    )
    .reset_index()
    .sort_values("unique_stops", ascending=False)
)

scope["Kontext"] = scope["context"]
scope["Betroffene Haltestellen"] = scope["unique_stops"]
scope["Betroffene Linien"] = scope["unique_lines"]
scope["Ø Empfohlener Puffer (s)"] = scope["mean_buffer"].round(1)
scope["Max. Puffer (s)"] = scope["max_buffer"]
show_df(
    scope[["Kontext", "Betroffene Haltestellen", "Betroffene Linien", "Ø Empfohlener Puffer (s)", "Max. Puffer (s)"]]
    .reset_index(drop=True)
)

## Key Findings

→ Vollständige Findings-Tabelle in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

`Präsentation`: **hot** = Kernbefund · **story** = gutes Narrativ · **—** = intern

| ID | Finding | Präsentation |
|:---|:---|:---:|
| F-REC-01 | **Vorhersagbarkeit = Strukturalität = Steuerbarkeit.** MAE 18.56s beweist: Delays folgen Mustern, sie sind kein Zufall. Was Muster hat, kann man designen. Das Modell macht aus einer Analyse-Frage eine Handlungsgrundlage. | **hot** |
| F-REC-02 | Das Modell identifiziert Stop-Linie-Kontext-Kombinationen mit systematisch hohem Delay (>60s vorhergesagt). Diese Kombinationen sind die präzise Grundlage für stopspezifische dwell_time-Kalibrierung — statt pauschaler 0/60s für alle. | **hot** |
| F-REC-03 | Kontextspezifische Muster: Schnee betrifft strukturell andere Stops als Events oder Rush-Hour. Eine einheitliche Pufferstrategie greift zu kurz — kontextsensitive Fahrpläne (Schneefahrplan, Eventfahrplan) sind die logische Konsequenz. | **story** |
| F-REC-04 | Empfohlene Puffergrößen sind Startpunkte, keine Garantien (→ F-SIM-03/04). Validierung erfordert operatives A/B-Testing: ausgewählte Stops erhalten angepasste dwell_time → Delay-Veränderung messen → kalibrieren. Das Modell liefert die Diagnose, der Betrieb die Dosis. | story |

## Export

In [ ]:
from pathlib import Path

img_dir = Path("../public/img")
img_dir.mkdir(parents=True, exist_ok=True)

# Re-build map figure explicitly so the export cell is self-contained
fig_export = go.Figure()
for ctx, col in ctx_colors.items():
    sub = flagged[flagged["context"] == ctx]
    if sub.empty:
        continue
    fig_export.add_trace(go.Scattermapbox(
        lat=sub["stop_lat"],
        lon=sub["stop_lon"],
        mode="markers",
        name=ctx,
        marker=dict(size=sub["bubble"], color=col, opacity=0.80),
        customdata=sub[["stop_short", "line_name", "pred_delay", "buffer_rec_s"]].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b> · L%{customdata[1]}<br>"
            "Pred. Delay: <b>%{customdata[2]:.0f}s</b><br>"
            "Buffer-Empfehlung: <b>+%{customdata[3]}s</b>"
            "<extra></extra>"
        ),
    ))
fig_export.update_layout(
    mapbox=dict(style="carto-positron", center=dict(lat=47.378, lon=8.540), zoom=11.5),
    title=dict(
        text="Fahrplan-Puffer-Empfehlungen nach Haltestelle und Kontext<br>"
             "<sup>Größe = empfohlener Puffer · Farbe = Betriebskontext</sup>",
        x=0, xanchor="left",
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    height=600,
    margin=dict(l=0, r=0, t=80, b=0),
)

out_path = img_dir / "scheduling-recommendations-map.html"
fig_export.write_html(str(out_path), include_plotlyjs="cdn")
print(f"✅ Scheduling recommendations map saved to {out_path}")